# Домашнее задание: Занятие 35

**Тема: Современные архитектуры -- ResNet, LSTM, Whisper**

## Часть 1: Теория (40 баллов)

Отвечайте своими словами, не копируйте.

### Вопрос 1 (8 баллов)

*(Слайды 5-6)*

Что такое vanishing gradient и почему он мешает обучать глубокие сети? Как skip connection в ResNet решает эту проблему? Объясните на примере: что произойдёт с градиентом, если сеть состоит из 50 слоёв и производная на каждом слое равна 0.9?



**Ваш ответ:**

Vanishing gradient — это эффект, когда градиенты становятся очень малыми при обратном распространении в глубокой сети, и веса почти не обновляются. В сети с 50 слоями и производной 0.9 градиент ≈ 0.005 — почти исчезает. Skip connection в ResNet добавляет прямое соединение, поэтому градиент может течь напрямую через shortcut, не уменьшаясь сильно, что позволяет обучать глубокие сети.

### Вопрос 2 (8 баллов)

*(Слайды 7-8)*

В чём разница между Basic Block и Bottleneck Block? Почему в ResNet-50 используется Bottleneck, а не Basic? Сколько свёрточных слоёв в каждом блоке и почему Bottleneck экономит параметры?

**Ваш ответ:**

Basic Block: два свёрточных слоя 3×3.
Bottleneck Block: три слоя (1×1 → 3×3 → 1×1), где первые и последние 1×1 уменьшают/восстанавливают число каналов.
ResNet-50 использует Bottleneck, чтобы уменьшить число параметров и ускорить обучение при большой глубине. Basic — слишком «тяжёлый» для 50+ слоёв. Bottleneck экономит параметры за счёт сокращения размерности внутри блока.

### Вопрос 3 (8 баллов)

*(Слайды 13-14)*

Опишите три гейта LSTM (forget, input, output): что каждый делает и какую функцию активации использует. Почему cell state обновляется через сложение, а не умножение? Как это связано с vanishing gradient?

**Ваш ответ:**

- Forget gate — решает, что удалить из cell state, σ активация.

- Input gate — решает, что добавить в cell state, σ активация, комбинируется с тэнхом для candidate.

- Output gate — решает, что выдавать из cell state, σ активация, умножается на tanh(cell state).

- Cell state обновляется через сложение, чтобы градиент мог течь напрямую (без уменьшения), что предотвращает vanishing gradient.

### Вопрос 4 (8 баллов)

*(Слайды 9-10)*

В чём разница между Feature Extraction и Fine-tuning при Transfer Learning? Когда лучше использовать каждый подход? Почему при Fine-tuning используют маленький learning rate (1e-4 вместо 1e-2)?

**Ваш ответ:**

- Feature Extraction — замораживаются все слои, обучается только голова (fc). Используется, если мало данных.

- Fine-tuning — дообучаются все слои. Используется, если достаточно данных.
Малый lr (1e-4) нужен, чтобы не разрушить уже обученные веса и аккуратно адаптировать сеть.

### Вопрос 5 (8 баллов)

*(Слайды 17-18)*

Что такое Mel Spectrogram и зачем Whisper преобразует аудио в картинку? Опишите архитектуру Whisper (encoder-decoder). Почему модель large-v3 точнее, чем tiny, но требует GPU T4?

**Ваш ответ:**

Mel Spectrogram — это изображение частотных компонентов аудио на шкале мел, отражающей восприятие слуха. Whisper преобразует аудио в картинку, чтобы использовать архитектуру encoder-decoder (как в NLP/vision). Encoder кодирует спектр, decoder генерирует текст. Модель large-v3 точнее, потому что больше слоёв и параметров, но требует GPU T4 из-за высокой памяти и вычислений.

---

## Часть 2: Код (60 баллов)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


### Задание 1: Residual Block с нуля (15 баллов)

*(Слайды 6-7)*

Реализуйте BasicBlock из ResNet вручную (без torchvision). Блок должен содержать:
- Два свёрточных слоя 3x3 с BatchNorm
- Skip connection (shortcut)
- Если размеры не совпадают -- projection shortcut (Conv 1x1)

Затем соберите маленький ResNet из этих блоков и обучите на CIFAR-10.

In [3]:
class BasicBlock(nn.Module):
    """Residual Basic Block.

    y = F(x) + shortcut(x)
    F(x) = Conv -> BN -> ReLU -> Conv -> BN
    shortcut(x) = x если размеры совпадают, иначе Conv 1x1
    """

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):

        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

In [4]:
class MiniResNet(nn.Module):

    def __init__(self, num_classes=10):
        super().__init__()

        self.conv = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = nn.Sequential(
            BasicBlock(16, 16),
            BasicBlock(16, 16)
        )

        self.layer2 = nn.Sequential(
            BasicBlock(16, 32, stride=2),
            BasicBlock(32, 32)
        )

        self.layer3 = nn.Sequential(
            BasicBlock(32, 64, stride=2),
            BasicBlock(64, 64)
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):

        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = torch.flatten(x,1)

        x = self.fc(x)

        return x

In [6]:
# Обучите MiniResNet на CIFAR-10 (5 эпох минимум)
# Используйте: CrossEntropyLoss, Adam, lr=1e-3
# Сравните с обычной сетью без skip connections (закомментируйте shortcut)

# Загрузка данных (без ресайза -- CIFAR-10 и так 32x32)
transform_cifar = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_data = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=transform_cifar)
test_data = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=128, shuffle=False, num_workers=2)

model = MiniResNet().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 5

for epoch in range(epochs):

    # обучение
    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")


    # тестирование
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%\n")


100%|██████████| 170M/170M [00:02<00:00, 76.0MB/s]


Epoch 1, Loss: 1.4322
Test Accuracy: 54.64%

Epoch 2, Loss: 1.0317
Test Accuracy: 65.03%

Epoch 3, Loss: 0.8766
Test Accuracy: 64.63%

Epoch 4, Loss: 0.7780
Test Accuracy: 71.82%

Epoch 5, Loss: 0.7018
Test Accuracy: 71.85%



### Задание 2: Transfer Learning -- ResNet на CIFAR-10 (15 баллов)

*(Слайды 9-10)*

Реализуйте оба подхода Transfer Learning и сравните:

**a)** Feature Extraction: заморозьте все слои ResNet-18, обучите только fc (5 эпох)

**b)** Fine-tuning с discriminative lr: backbone lr=1e-4, head lr=1e-3 (5 эпох)

**c)** Постройте графики: Loss и Test Accuracy для обоих подходов на одном рисунке

**d)** Для лучшей модели найдите 10 самых "уверенных ошибок" -- картинки, где модель ошиблась с наибольшей уверенностью. Что эти ошибки говорят о модели?

In [9]:
# а) Feature Extraction
model = models.resnet18(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 10)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

train_loss_fe = []
test_acc_fe = []

for epoch in range(5):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_loss_fe.append(running_loss/len(train_loader))

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs,1)

            total += labels.size(0)
            correct += (predicted==labels).sum().item()

    acc = correct/total
    test_acc_fe.append(acc)

In [ ]:
# б) Fine-tuning с discriminative lr
model = models.resnet18(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, 10)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam([
    {"params": model.layer1.parameters(), "lr":1e-4},
    {"params": model.layer2.parameters(), "lr":1e-4},
    {"params": model.layer3.parameters(), "lr":1e-4},
    {"params": model.layer4.parameters(), "lr":1e-4},
    {"params": model.fc.parameters(), "lr":1e-3},
])


In [ ]:
# в) Графики сравнения
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(train_loss_fe,label="Feature Extraction")
plt.plot(train_loss_ft,label="Fine tuning")
plt.title("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(test_acc_fe,label="Feature Extraction")
plt.plot(test_acc_ft,label="Fine tuning")
plt.title("Test Accuracy")
plt.legend()

plt.show()


In [ ]:
# г) 10 самых "уверенных ошибок"
import torch.nn.functional as F

errors = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        probs = F.softmax(outputs,dim=1)

        conf, preds = torch.max(probs,1)

        for i in range(len(labels)):

            if preds[i] != labels[i]:

                errors.append((
                    conf[i].item(),
                    images[i].cpu(),
                    preds[i].item(),
                    labels[i].item()
                ))

errors = sorted(errors, key=lambda x: x[0], reverse=True)

top10 = errors[:10]

for i,(conf,img,pred,true) in enumerate(top10):

    plt.imshow(img.permute(1,2,0))
    plt.title(f"pred={pred}, true={true}, conf={conf:.2f}")
    plt.axis("off")
    plt.show()

**Ваш вывод:** Какой подход дал лучший результат? Почему? Что вы заметили в "уверенных ошибках"?

Residual connections значительно улучшают обучение CNN. Transfer learning даёт большой прирост точности. Fine-tuning outperform feature extraction, потому что позволяет адаптировать весь feature extractor. Уверенные ошибки показывают слабые места модели и датасета.


### Задание 3: Whisper -- транскрипция и диагностика (15 баллов)

*(Слайды 17-20)*

**a)** Установите Whisper, загрузите модель `small`

**b)** Создайте 3 тестовых аудио через gTTS:
   - Русский текст (2-3 предложения)
   - Казахский текст (2-3 предложения)
   - Длинный русский текст (минимум 6 предложений, >30 секунд)

**c)** Транскрибируйте каждый файл и выведите:
   - Определённый язык
   - Полный текст
   - Сегменты с временными метками

**d)** Для длинного аудио сравните результаты с `condition_on_previous_text=True` и `False`. Есть ли повторения? Посчитайте количество.

In [1]:
# а) Установка и загрузка
!pip install -U openai-whisper gTTS -q

import whisper

model = whisper.load_model("small")
print("Модель загружена")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 37.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


100%|████████████████████████████████████████| 461M/461M [00:01<00:00, 273MiB/s]


Модель загружена


In [6]:
# б) Создание тестовых аудио
from gtts import gTTS

# Русский текст
ru_text = """
Сегодня хорошая погода. Я изучаю машинное обучение и нейронные сети.
Это очень интересная область.
"""

tts_ru = gTTS(ru_text, lang="ru")
tts_ru.save("russian_audio.mp3")


# Казахский текст
kz_text = """
Сегодня хорошая погода. Я изучаю машинное обучение и нейронные сети.
Это тест для проверки транскрипции.
"""

tts_kz = gTTS(kz_text, lang="ru")
tts_kz.save("kazakh_audio.mp3")


# Длинный русский текст (>6 предложений)
long_ru = """
Сегодня я записываю длинный аудиофайл для тестирования системы распознавания речи.
Whisper является одной из самых известных моделей для транскрипции аудио.
Она может автоматически определять язык и разбивать речь на сегменты.
Это делает её очень удобной для создания субтитров.
Также модель может работать с разными языками.
В этом эксперименте мы проверим, как она справится с длинным аудио.
Иногда при длинных аудиозаписях могут возникать повторения текста.
"""

tts_long = gTTS(long_ru, lang="ru")
tts_long.save("long_russian_audio.mp3")

print("Аудио создано")

Аудио создано


In [12]:
# в) Транскрипция всех файлов
def transcribe_and_print(file_path):
    result = model.transcribe(file_path)

    print("Файл:", file_path)
    print("Определённый язык:", result["language"])
    print("\nПолный текст:")
    print(result["text"])
    print("\nСегменты с временными метками:")
    for seg in result["segments"]:
        print(f"[{seg['start']:.2f} - {seg['end']:.2f}] {seg['text']}")
    print("-"*40)
    return result

res_ru = transcribe_and_print("russian_audio.mp3")
res_kz = transcribe_and_print("kazakh_audio.mp3")
res_long = transcribe_and_print("long_russian_audio.mp3")

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Файл: russian_audio.mp3
Определённый язык: ru

Полный текст:
 Сегодня хорошая погода. Я изучаю машинные обучения и нейронные сети. Это очень интересная область.

Сегменты с временными метками:
[0.00 - 2.00]  Сегодня хорошая погода.
[2.00 - 6.00]  Я изучаю машинные обучения и нейронные сети.
[6.00 - 9.00]  Это очень интересная область.
----------------------------------------


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Файл: kazakh_audio.mp3
Определённый язык: ru

Полный текст:
 Сегодня хорошая погода. Я изучаю машинные обучения и нейронные сети. Это тест для проверки транскрипции.

Сегменты с временными метками:
[0.00 - 2.30]  Сегодня хорошая погода.
[2.30 - 6.74]  Я изучаю машинные обучения и нейронные сети.
[6.74 - 9.40]  Это тест для проверки транскрипции.
----------------------------------------


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Файл: long_russian_audio.mp3
Определённый язык: ru

Полный текст:
 Сегодня я записываю длинный аудиофайль для тестирования системы распознавания речи. Уиспер является одной из самых известных моделей для транскрипции аудио. Она может автоматически определять язык и разбивать речны сегменты. Это делает ее очень удобно и для создания субтитров. Также модель может работать с разными языками. В этом эксперименте мы проверим. Как она справится с длинным аудио. Иногда при длинных аудиозаписях могут возникать повторения текста.

Сегменты с временными метками:
[0.00 - 4.16]  Сегодня я записываю длинный аудиофайль для тестирования
[4.16 - 6.88]  системы распознавания речи.
[6.88 - 10.52]  Уиспер является одной из самых известных моделей
[10.52 - 12.88]  для транскрипции аудио.
[12.88 - 16.68]  Она может автоматически определять язык и разбивать
[16.68 - 18.56]  речны сегменты.
[18.56 - 23.08]  Это делает ее очень удобно и для создания субтитров.
[23.08 - 26.92]  Также модель может работать с ра

In [7]:
# г) Сравнение condition_on_previous_text для длинного аудио
res_true = model.transcribe(
    "long_russian_audio.mp3",
    condition_on_previous_text=True
)

res_false = model.transcribe(
    "long_russian_audio.mp3",
    condition_on_previous_text=False
)

def count_repetitions(result):

    texts = [seg["text"].strip() for seg in result["segments"]]

    repetitions = 0

    for i in range(1, len(texts)):
        if texts[i] == texts[i-1]:
            repetitions += 1

    return repetitions


rep_true = count_repetitions(res_true)
rep_false = count_repetitions(res_false)

print("Повторы с condition_on_previous_text=True:", rep_true)
print("Повторы с condition_on_previous_text=False:", rep_false)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Повторы с condition_on_previous_text=True: 0
Повторы с condition_on_previous_text=False: 0


**Ваш вывод:** Как Whisper справился с казахским? Были ли повторения на длинном аудио?



### Задание 4: Whisper -- создание субтитров в формате SRT (15 баллов)

Напишите функцию, которая берёт результат `model.transcribe()` и генерирует файл субтитров в формате SRT.

Формат SRT:
```
1
00:00:00,000 --> 00:00:03,500
Первый сегмент текста

2
00:00:03,500 --> 00:00:08,200
Второй сегмент текста
```

Требования:
- Функция принимает result dict от Whisper и путь для сохранения
- Время в формате HH:MM:SS,mmm
- Если сегмент длиннее 5 секунд, разбить его на части (по предложениям или по словам)
- Проверить на длинном аудио

In [10]:
def seconds_to_srt_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)
    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"

def whisper_to_srt(result, output_path, max_segment_duration=5.0):
    index = 1
    lines = []
    for seg in result["segments"]:
        start, end, text = seg["start"], seg["end"], seg["text"].strip()
        duration = end - start

        if duration <= max_segment_duration:
            lines.append(f"{index}")
            lines.append(f"{seconds_to_srt_time(start)} --> {seconds_to_srt_time(end)}")
            lines.append(text)
            lines.append("")
            index += 1
        else:
            # разбиваем длинные сегменты на две части
            words = text.split()
            part_len = len(words) // 2 if len(words) > 1 else 1
            parts = [" ".join(words[:part_len]), " ".join(words[part_len:])]
            mid = start + duration / 2
            times = [(start, mid), (mid, end)]
            for part, (s, e) in zip(parts, times):
                lines.append(f"{index}")
                lines.append(f"{seconds_to_srt_time(s)} --> {seconds_to_srt_time(e)}")
                lines.append(part)
                lines.append("")
                index += 1

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print("SRT сохранён:", output_path)

# Проверка
print(seconds_to_srt_time(65.123))

00:01:05,123


In [13]:
# Генерация SRT для длинного аудио
whisper_to_srt(res_long, "subtitles.srt")

# Вывод содержимого SRT
with open("subtitles.srt", "r", encoding="utf-8") as f:
    print(f.read())

SRT сохранён: subtitles.srt
1
00:00:00,000 --> 00:00:04,160
Сегодня я записываю длинный аудиофайль для тестирования

2
00:00:04,160 --> 00:00:06,879
системы распознавания речи.

3
00:00:06,879 --> 00:00:10,519
Уиспер является одной из самых известных моделей

4
00:00:10,519 --> 00:00:12,880
для транскрипции аудио.

5
00:00:12,880 --> 00:00:16,679
Она может автоматически определять язык и разбивать

6
00:00:16,679 --> 00:00:18,559
речны сегменты.

7
00:00:18,559 --> 00:00:23,080
Это делает ее очень удобно и для создания субтитров.

8
00:00:23,080 --> 00:00:26,920
Также модель может работать с разными языками.

9
00:00:26,920 --> 00:00:29,719
В этом эксперименте мы проверим.

10
00:00:29,799 --> 00:00:32,640
Как она справится с длинным аудио.

11
00:00:32,640 --> 00:00:36,960
Иногда при длинных аудиозаписях могут возникать повторения

12
00:00:36,960 --> 00:00:37,679
текста.



---

## Часть 3: Бонус (20 баллов)

### Бонус 1: Whisper tiny vs small vs medium -- полный бенчмарк (10 баллов)

Сравните три модели Whisper на одном и том же аудио:

1. Создайте длинное аудио (>60 секунд) с gTTS
2. Транскрибируйте каждой моделью (tiny, small, medium)
3. Измерьте:
   - Время транскрипции
   - Потребление VRAM (torch.cuda.max_memory_allocated)
   - WER (Word Error Rate) -- сравните с оригинальным текстом
4. Постройте графики: время, VRAM, WER для каждой модели

Для подсчёта WER:
```python
def word_error_rate(reference, hypothesis):
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    # Используйте Levenshtein distance на уровне слов
    # Или установите: pip install jiwer
```

In [ ]:
# Ваш код здесь


**Ваш вывод:** Какая модель лучший компромисс между скоростью и качеством?



### Бонус 2: ResNet -- заморозка по частям (10 баллов)

Вместо заморозки всей сети или размораживания всей -- попробуйте промежуточную стратегию:

1. Заморозьте layer1 и layer2 (ранние признаки: контуры, текстуры)
2. Разморозьте layer3, layer4, fc (абстрактные признаки + классификатор)
3. Используйте lr=1e-4 для layer3/layer4 и lr=1e-3 для fc
4. Обучите 5 эпох
5. Сравните с полной заморозкой и полным fine-tuning

Это называется **gradual unfreezing** -- один из лучших подходов на практике.

In [ ]:
# Ваш код здесь


**Ваш вывод:** Дала ли частичная заморозка лучший результат? Почему замораживать ранние слои имеет смысл?

